In [ ]:
# =========================
# STEP 1: LOAD DATA & SETUP
# Description: 
# This script loads the EPC dataset and iterates through each building folder.
# For each building, it reads the "annual_savings.csv" file and maps values
# (ENERGY, CARBON, COST, SAVINGS, etc.) into the corresponding columns in the EPC dataset.
# The mapping is based on the LABEL (e.g., BASELINE, WINDOWS, etc.) and column names.
# Finally, it outputs an updated EPC dataset with all values filled in.

import pandas as pd
import os

# File paths
epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
base_folder = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

# Load EPC dataset
epc_df = pd.read_csv(epc_path)

# Ensure BUILDING_REFERENCE_NUMBER is string for matching
epc_df["BUILDING_REFERENCE_NUMBER"] = epc_df["BUILDING_REFERENCE_NUMBER"].astype(str)

print("EPC dataset loaded. Number of buildings:", len(epc_df))

EPC dataset loaded. Number of buildings: 117


In [2]:
# =========================
# STEP 2: DEFINE LABEL + COLUMN MAPPING
# Description:
# Maps LABEL values in annual_savings.csv to EPC column prefixes

label_map = {
    "BASELINE": "BASELINE",
    "WINDOWS": "WINDOWS",
    "WALLS": "WALLS",
    "ROOF": "ROOF",
    "FAB": "FAB",  # FAB maps to FAB columns
    "HP": "HP",
    "SOLAR_THERMAL": "SOLAR_THERMAL",
    "PV": "PV",
    "BATTERY": "BATTERY"
}

# Columns to transfer
value_columns = [
    "ENERGY",
    "CARBON",
    "COST_FIXED",
    "COST_FLEX",
    "ENERGY_SAVINGS",
    "CARBON_SAVINGS",
    "COST_SAVINGS_FIXED",
    "COST_SAVINGS_FLEX"
]

print("Mappings defined.")

Mappings defined.


In [3]:
# =========================
# STEP 3: PROCESS EACH BUILDING
# Description:
# Loops through each building folder, reads annual_savings.csv,
# and writes values into the EPC dataframe.

missing_files = []
processed_count = 0

for idx, row in epc_df.iterrows():
    building_id = row["BUILDING_REFERENCE_NUMBER"]
    
    file_path = os.path.join(base_folder, building_id, "TOTAL", "annual_savings.csv")
    
    if not os.path.exists(file_path):
        missing_files.append(building_id)
        continue
    
    try:
        savings_df = pd.read_csv(file_path)
        
        # Loop through each row in annual_savings.csv
        for _, s_row in savings_df.iterrows():
            label = s_row["LABEL"]
            
            if label not in label_map:
                continue
            
            prefix = label_map[label]
            
            # Map each value column
            for col in value_columns:
                epc_col = f"{prefix}_{col}"
                
                if epc_col in epc_df.columns:
                    epc_df.at[idx, epc_col] = s_row[col]
        
        processed_count += 1
    
    except Exception as e:
        print(f"Error processing {building_id}: {e}")

print(f"Processed buildings: {processed_count}")
print(f"Missing files: {len(missing_files)}")

Processed buildings: 117
Missing files: 0


In [4]:
# =========================
# STEP 4: SAVE UPDATED DATASET
# Description:
# Saves the updated EPC dataset to a new CSV file.

output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_with_annual_data.csv"

epc_df.to_csv(output_path, index=False)

print("Updated EPC dataset saved to:")
print(output_path)

Updated EPC dataset saved to:
/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_with_annual_data.csv


In [5]:
# =========================
# STEP 5: OPTIONAL VALIDATION CHECK
# Description:
# Quickly check a sample building to verify values were transferred correctly.

sample_id = epc_df["BUILDING_REFERENCE_NUMBER"].iloc[0]
sample_row = epc_df[epc_df["BUILDING_REFERENCE_NUMBER"] == sample_id]

print("Sample building ID:", sample_id)
print(sample_row.T.head(30))

Sample building ID: 1001829067
                                                                                0
BUILDING_REFERENCE_NUMBER                                              1001829067
OSG_REFERENCE_NUMBER                                                  122009933.0
ADDRESS1                                                     19 CULCREUCH AVENUE 
ADDRESS2                                                                  FINTRY 
ADDRESS3                                                                 GLASGOW 
POSTCODE                                                                  G63 0YB
LATITUDE                                                                  56.0557
LONGITUDE                                                                 -4.2233
ORIENTATION                                                                   100
ORIENTATION_ESPR_ROT                                                          170
INSPECTION_DATE                                                    